# Préparation de l'espace de travail

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display, clear_output

In [ ]:
data_licences = pd.read_parquet("data/data_licences/data_licences.parquet")

# Graphiques

In [ ]:
def plot_licencies_age_plotly(df, age):
    """
    Renvoie un graphique d'évolution des effectifs de licenciés par âge et par sport.
    
    Paramètres
    ---------
    df (pd.DataFrame) : data frame contenant les données de licences
    age (str) : âge sélectionné
    
    Output
    ---------
    graphique
    """
    
    df_clean = df[df["code_sport"] != "DIV"]
    df_filtre = df_clean[df_clean["age"] == age]

    table = (
        df_filtre.groupby(["annee", "code_sport"])["licences_annuelles"]
        .sum()
        .unstack()
        .sort_index()
    )

    # Tracé
    fig = px.line(
        table,
        x=table.index,
        y=table.columns,
        color="code_sport",
        color_discrete_sequence=px.colors.qualitative.Alphabet,
        markers=True,
        labels={
            "x": "Année",
            "value": "Nombre de licenciés",
            "code_sport": "Sport"
        },
        title=f"Évolution du nombre de licenciés de {age} ans par sport"
    )

    fig.update_layout(width=1100, height=600)
    fig.show()

plot_licencies_age_plotly(data_licences, "6")


In [ ]:
def plot_licencies_age_tranche_plotly(df, tranche):
    """ Renvoie un graphique d'évolution des effectifs de licenciés par tranche d'âge et par sport.

    Paramètres
    ---------
    df (pd.DataFrame) : data frame contenant les données de licences
    tranche (str) : tranche d'âge sélectionnée, indexée par des lettres (par exemple "a" pour 1 à 4 ans)
    
    Output
    ---------
    graphique
    """

    df_clean = df[df["code_sport"] != "DIV"]
    df_filtre = df_clean[df_clean["tranche_age"].str[0] == tranche].copy()

    # S'assurer que l'année est triée
    df_filtre = df_filtre.sort_values(["annee", "code_sport"])

    # Récupérer la tranche lisible
    try:
        tranche_label = df_filtre["tranche_age"].str[4:].unique()[0]
    except:
        tranche_label = ""

    # Aggrégation
    table = (
        df_filtre.groupby(["annee", "code_sport"])["licences_annuelles"]
          .sum()
          .unstack()
          .sort_index()
    )

    # Tracé
    fig = px.line(
        table,
        x=table.index,
        y=table.columns,
        markers=True,
        color_discrete_sequence=px.colors.qualitative.Alphabet,
        labels={
            "x": "Année",
            "value": "Nombre de licenciés",
            "code_sport": "Sport"
        },
        title=f"Évolution du nombre de licenciés {tranche_label} ans par sport"
    )

    fig.update_layout(width=1200, height=650)
    fig.show()


plot_licencies_age_tranche_plotly(data_licences, "a")


In [ ]:
def sport_age_decomposition(df, annee):
    """
    Renvoie un graphique décomposant les effectifs de licenciés par tranche d'âge (grande tranche) et par sport pour une année.

    Paramètres
    ---------
    df (pd.DataFrame) : data frame contenant les données de licences
    annee (int) : année sélectionnée
    
    Output
    ---------
    graphique
    """

    df_clean = df[df["annee"] == annee]

    df_pivot = df_clean.pivot_table(
        index='code_sport',
        columns='grande_tranche_age',
        values='licences_annuelles',
        aggfunc='sum',
        fill_value=0
    )

    df_prop = df_pivot.div(df_pivot.sum(axis=1), axis=0)
    
    df_prop.plot(kind='barh', stacked=True, figsize=(10,6))
    
    plt.title(f"Répartition proportionnelle des licenciés par sport et tranche d'âge - {annee}")
    plt.xlabel("Proportion de licenciés")
    plt.ylabel("Sport")
    plt.legend(title="Tranche d'âge", bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.xlim(0,1)  
    plt.tight_layout()
    plt.show()

sport_age_decomposition(data_licences, 2016)

In [ ]:
def sport_age_decomposition_plotly(df, annee):
    """
    Renvoie un graphique décomposant les effectifs de licenciés par tranche d'âge (grande tranche) et par sport pour une année.

    Paramètres
    ---------
    df (pd.DataFrame) : data frame contenant les données de licences
    annee (int) : année sélectionnée
    
    Output
    ---------
    graphique
    """

    df_clean = df[df["annee"] == annee]

    df_pivot = df_clean.pivot_table(
        index='code_sport',
        columns='grande_tranche_age',
        values='licences_annuelles',
        aggfunc='sum',
        fill_value=0
    )

    df_prop = df_pivot.div(df_pivot.sum(axis=1), axis=0)

    df_long = df_prop.reset_index().melt(
        id_vars="code_sport",
        var_name="tranche_age",
        value_name="proportion"
    )

    # Passer en pourcentage
    df_long["proportion"] = df_long["proportion"] * 100

    # --- Construire le mapping des couleurs ---
    tranches = sorted(df_long["tranche_age"].unique())  # liste des tranches d'âge

    # Si "NR" existe, on l'isole
    palette_map = {}

    if "NR - Non réparti" in tranches:
        tranches_no_nr = [t for t in tranches if t != "NR - Non réparti"]
    else:
        tranches_no_nr = tranches

    # Nombre de couleurs à générer pour les tranches (hors NR)
    n = len(tranches_no_nr)

    # Générer n couleurs dans Plasma_r, du jaune (min) au violet (max)
    colors = px.colors.sample_colorscale(
        px.colors.sequential.Plasma_r,
        [i/(n-1) for i in range(n)] if n > 1 else [0.5]
    )

    # Assigner les couleurs aux tranches (hors NR)
    for tranche, col in zip(tranches_no_nr, colors):
        palette_map[tranche] = col

    # Assigner NR en noir si présent
    if "NR - Non réparti" in tranches:
        palette_map["NR - Non réparti"] = "black"

    # --- Graphique ---
    fig = px.bar(
        df_long,
        x="proportion",
        y="code_sport",
        color="tranche_age",
        color_discrete_map=palette_map,
        orientation="h",
        barmode="stack",
        labels={
            "proportion": "Proportion de licenciés (%)",
            "code_sport": "Sport",
            "tranche_age": "Tranche d'âge"
        },
        title=f"Répartition proportionnelle des licenciés par sport et tranche d'âge – {annee}"
    )

    fig.update_xaxes(ticksuffix="%")
    fig.update_layout(
        width=1000,
        height=600,
        xaxis=dict(range=[0, 100])
    )

    fig.show()


sport_age_decomposition_plotly(data_licences, 2016)


In [ ]:
def sport_age_decomposition_plotly(df, annee):
    """ Renvoie un graphique décomposant les effectifs de licenciés par tranche d'âge (tranche d'âge fine) et par sport pour une année.

    Paramètres
    ---------
    df (pd.DataFrame) : data frame contenant les données de licences
    annee (int) : année sélectionnée
    
    Output
    ---------
    graphique
    """

    df_clean = df[df["annee"] == annee]

    df_pivot = df_clean.pivot_table(
        index='code_sport',
        columns='tranche_age',
        values='licences_annuelles',
        aggfunc='sum',
        fill_value=0
    )

    # Proportion par ligne
    df_prop = df_pivot.div(df_pivot.sum(axis=1), axis=0).fillna(0)

    # Format long
    df_long = df_prop.reset_index().melt(
        id_vars="code_sport",
        var_name="tranche_age",
        value_name="proportion"
    )

    # Supprimer tranches vides ou NaN
    df_long = df_long[df_long["tranche_age"].notna()]
    df_long["tranche_age"] = df_long["tranche_age"].astype(str)

    # Passer en pourcentage
    df_long["proportion"] = df_long["proportion"] * 100


    # Couleurs
    tranches = sorted(df_long["tranche_age"].unique())
    
    # Séparer NR
    tranches_no_nr = [t for t in tranches if t != "NR - Non réparti"]

    n = len(tranches_no_nr)

    # Générer n couleurs dans Plasma_r
    colors = px.colors.sample_colorscale(
        px.colors.sequential.Plasma_r,
        [i/(n-1) for i in range(n)] if n>1 else [0.5]
    )

    palette_map = {t: c for t, c in zip(tranches_no_nr, colors)}
    if "NR - Non réparti" in tranches:
        palette_map["NR - Non réparti"] = "black"
    

    # --- Définir l'ordre des tranches ---
    # Trier alphabétiquement sauf NR
    tranches_ord = sorted(tranches_no_nr)
    # Ajouter NR à la fin si présent
    if "NR - Non réparti" in tranches:
        tranches_ord.append("NR - Non réparti")


    # Tracé
    fig = px.bar(
        df_long,
        x="proportion",
        y="code_sport",
        color="tranche_age",
        color_discrete_map=palette_map,
        category_orders={"tranche_age": tranches_ord},
        orientation="h",
        barmode="stack",
        labels={
            "proportion": "Proportion de licenciés (%)",
            "code_sport": "Sport",
            "tranche_age": "Tranche d'âge"
        },
        title=f"Répartition proportionnelle des licenciés par sport et tranche d'âge – {annee}"
    )

    # Axe X en pourcentage
    fig.update_xaxes(ticksuffix="%")
    fig.update_layout(
        width=1000,
        height=600,
        xaxis=dict(range=[1, 100]))

    fig.show()

sport_age_decomposition_plotly(data_licences, 2016)


In [ ]:
# --- Préparer les options pour les widgets ---
ages = sorted(data_licences["tranche_age"].dropna().unique())
sports = sorted(data_licences["code_sport"].dropna().unique())
annees = sorted(data_licences["annee"].dropna().unique())

age_widget = widgets.Dropdown(options=ages, description="Âge :", value=ages[0])
sport_widget = widgets.Dropdown(options=["all"] + sports, description="Sport :", value="all")
annee_widget = widgets.Dropdown(options=["all"] + list(annees), description="Année :", value="all")

# --- Fonction pour filtrer et préparer les données ---
def filter_data(df, age, sport, annee):
    df_filtered = df[df["tranche_age"] == age]
    
    if sport != "all":
        df_filtered = df_filtered[df_filtered["code_sport"] == sport]
        
    if annee != "all":
        df_filtered = df_filtered[df_filtered["annee"] == annee]
    
    table = (
        df_filtered.groupby(["annee", "code_sport"])["licences_annuelles"]
        .sum()
        .reset_index()
    )
    return table

# --- Fonction pour tracer avec Plotly ---
def plot_licencies_age_interactive(age, sport, annee):
    df_plot = filter_data(data_licences, age, sport, annee)
    if df_plot.empty:
        fig = px.line(title=f"Aucune donnée pour l'âge {age} ans avec ce filtre")
    else:
        fig = px.line(
            df_plot,
            x="annee",
            y="licences_annuelles",
            color="code_sport",
            markers=True,
            title=f"Évolution des licenciés de {age} ans par sport"
        )
    fig.update_layout(
        xaxis_title="Année",
        yaxis_title="Nombre de licenciés",
        legend_title="Sport",
        template="plotly_white"
    )
    fig.show()

# --- Callback pour mettre à jour le graphique ---
def update_graph(change=None):
    clear_output(wait=True)
    display(age_widget, sport_widget, annee_widget)
    plot_licencies_age_interactive(age_widget.value, sport_widget.value, annee_widget.value)

# --- Lier les widgets ---
age_widget.observe(update_graph, names='value')
sport_widget.observe(update_graph, names='value')
annee_widget.observe(update_graph, names='value')

# --- Affichage initial ---
display(age_widget, sport_widget, annee_widget)
update_graph()
